<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/flexai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FlexAI

[FlexAI](https://flex.ai) serves open-weight models behind an OpenAI-compatible API, so it works
with LlamaIndex through the `OpenAILike` and `OpenAILikeEmbedding` classes.

Sign up at [flex.ai](https://flex.ai) to get an API key, and see the
[documentation](https://docs.flex.ai) for details. The `/v1/models` endpoint returns the models
that are currently served, and the model name to pass below is the `id` from that response.

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index llama-index-llms-openai-like llama-index-embeddings-openai-like

## Setup

Set `FLEXAI_API_KEY` in your environment, or pass it to the constructor as `api_key`.

In [ ]:
import os

os.environ["FLEXAI_API_KEY"] = "<your-api-key>"

In [ ]:
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model="DeepSeek-V4-Flash-0731",
    api_base="https://api.flex.ai/v1",
    api_key=os.environ["FLEXAI_API_KEY"],
    context_window=1048576,
    is_chat_model=True,
    is_function_calling_model=True,
)

#### Call `complete` with a prompt

In [ ]:
resp = llm.complete("Name the capital of France in one word.")
print(resp)

Paris


#### Call `chat` with a list of messages

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="You answer in one short sentence."),
    ChatMessage(role="user", content="What is an open-weight model?"),
]
resp = llm.chat(messages)
print(resp)

assistant: An open-weight model is an AI whose trained parameters (weights) are publicly released for anyone to use, modify, or study.


## Streaming

Both `stream_complete` and `stream_chat` are supported.

In [ ]:
resp = llm.stream_chat(
    [ChatMessage(role="user", content="Count from 1 to 5, comma separated.")]
)
for r in resp:
    print(r.delta, end="")

1, 2, 3, 4, 5

## Tool calling

Models that support tool calling are marked with the `tools` feature in the `/v1/models`
response. Pass `is_function_calling_model=True` when constructing the LLM, as above.

In [ ]:
from llama_index.core.tools import FunctionTool


def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)

resp = llm.chat_with_tools(
    [multiply_tool], user_msg="What is 1234 * 5678? Use the tool."
)
print(llm.get_tool_calls_from_response(resp))

[ToolSelection(tool_id='chatcmpl-tool-a91456423a7da877', tool_name='multiply', tool_kwargs={'a': 1234, 'b': 5678})]


## Embeddings

FlexAI serves embedding models on the same base URL.

In [ ]:
from llama_index.embeddings.openai_like import OpenAILikeEmbedding

embed_model = OpenAILikeEmbedding(
    model_name="bge-m3",
    api_base="https://api.flex.ai/v1",
    api_key=os.environ["FLEXAI_API_KEY"],
)

embedding = embed_model.get_text_embedding(
    "Open-weight models on a shared API."
)
print(len(embedding))

1024
